In [1]:
# Convertir cette classe écrite "à la main" en @dataclass équivalente :
# Bonus : ajouter un champ created_at initialisé à datetime.now() par défaut.

# class Order:
#     def __init__(self, id, customer, items):
#         self.id = id
#         self.customer = customer
#         self.items = items

#     def __repr__(self):
#         return f"Order(id={self.id!r}, customer={self.customer!r}, items={self.items!r})"

#     def __eq__(self, other):
#         if not isinstance(other, Order):
#             return NotImplemented
#         return (self.id, self.customer, self.items) == (other.id, other.customer, other.items)

from dataclasses import dataclass, field
from datetime import datetime
from typing import Any

@dataclass
class Order:
    id: int
    customer: str
    items: list[Any]
    created_at: datetime = field(default_factory=datetime.now)

In [2]:
# Concevoir un @dataclass User avec :
# - id: int
# - email: str
# - tags: list[str] par défaut [], non comparé (compare=False).
# - password_hash: str caché du __repr__.
# - created_at: datetime initialisé à datetime.now() à chaque instanciation.

# Vérifier :
# - Le repr ne montre pas password_hash.
# - User(1, "a@b") == User(1, "a@b", tags=["x"]) est vrai (tags ignorés).
# - Deux utilisateurs créés à un instant différent ne sont pas égaux si on ne marque pas created_at avec compare=False.

from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class User:
    id: int
    email: str
    tags: list[str] = field(default_factory=list, compare=False)
    password_hash: str = field(default="", repr=False)
    created_at: datetime = field(default_factory=datetime.now)
    # created_at: datetime = field(default_factory=datetime.now, compare=False)

In [3]:
# Concevoir une @dataclass(frozen=True) Config avec :

# host: str
# port: int
# tls: bool = True

# Démontrer :

# Une modification directe (config.port = 80) lève FrozenInstanceError.
# replace(config, port=8443) produit une nouvelle instance avec port=8443.
# Un set de Config élimine les doublons (puisqu'immuable et hashable).

from dataclasses import dataclass, replace

@dataclass(frozen=True)
class Config:
    host: str
    port: int
    tls: bool = True

config = Config(host="me", port=8000)
# >>> config.port = 80
# dataclasses.FrozenInstanceError: cannot assign to field 'port'

newInstance = replace(config, port=8443)
print(id(newInstance), id(config))
# 133612898160704 133612898056960

config2 = Config(host="me", port=8000)
print(set([config, config2]))
# {Config(host='me', port=8000, tls=True)}

config3 = Config(host="me", port=80)
print(set([config, config2, config3]))
# {Config(host='me', port=80, tls=True), Config(host='me', port=8000, tls=True)}

2422736813040 2422711484096
{Config(host='me', port=8000, tls=True)}
{Config(host='me', port=80, tls=True), Config(host='me', port=8000, tls=True)}


In [4]:
# Définir une classe abstraite Notifier avec :

# @abstractmethod def send(self, recipient: str, message: str) -> bool
# @property + @abstractmethod def name(self) -> str (propriété abstraite)

# Implémenter deux notifications concrètes :

# EmailNotifier(Notifier), SmsNotifier(Notifier)

# Écrire une fonction broadcast(notifiers: list[Notifier], recipient, message)
# qui appelle send sur chaque notifier et collecte les résultats dans un dict {notifier.name: bool}.

from dataclasses import dataclass
from abc import ABC, abstractmethod

@dataclass(frozen=True)
class Notifier(ABC):
    @abstractmethod
    def send(self, recipient: str, message: str) -> bool:
        pass

    @property
    @abstractmethod
    def name(self) -> str:
        pass

class EmailNotifier(Notifier):
    def send(self, recipient: str, message: str) -> bool:
        pass
        print(f"Sending email to {recipient}: '{message}'")
        return True

    @property
    def name(self) -> str:
        return "EmailNotifier"
e = EmailNotifier()

class SmsNotifier(Notifier):
    def send(self, recipient: str, message: str) -> bool:
        print(f"Sending SMS to {recipient}: '{message}'")
        return True

    @property
    def name(self) -> str:
        return "SmsNotifier"
s = SmsNotifier()

def broadcast(notifiers: list[Notifier], recipient: str, message: str) -> dict[str, bool]:
    results: dict[str, bool] = {}
    for n in notifiers:
        succeeded = n.send(recipient, message)
        results[n.name] = succeeded
    return results
broadcast_result = broadcast([e, s], "ptdr", "jpp")
print(f"{broadcast_result=}")

Sending email to ptdr: 'jpp'
Sending SMS to ptdr: 'jpp'
broadcast_result={'EmailNotifier': True, 'SmsNotifier': True}


In [5]:
# Reprendre la classe Shape abstraite et créer :

# Circle(Shape) avec radius et area().
# Rectangle(Shape) avec width, height et area().
# Triangle(Shape) avec base, height et area().
# Toutes en @dataclass(frozen=True). Écrire un test qui range une liste de shapes par aire croissante via sorted(shapes, key=lambda s: s.area()).

from dataclasses import dataclass
from abc import ABC, abstractmethod
from math import pi

@dataclass(frozen=True)
class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        ...

@dataclass(frozen=True)
class Circle(Shape):
    radius: int

    def area(self) -> float:
        return self.radius * 2 * pi

@dataclass(frozen=True)
class Rectangle(Shape):
    width: int
    height: int

    def area(self) -> float:
        return self.width * self.height / 2

@dataclass(frozen=True)
class Triangle(Shape):
    base: int
    height: int

    def area(self) -> float:
        return self.base * self.height / 2

shapes = [Circle(10), Rectangle(20, 30), Triangle(2, 4)]
# >>> sorted(shapes, key=lambda s: s.area())

In [6]:
# Modéliser un système de paiements en utilisant tous les outils du module :

# @dataclass(frozen=True) Money avec amount: int (en cents) et currency: str.
# Classe abstraite PaymentMethod(ABC) :
# @abstractmethod def charge(self, amount: Money) -> str (renvoie un id de transaction).
# @abstractmethod def refund(self, transaction_id: str) -> bool.
# Trois implémentations : CreditCardPayment, BankTransferPayment, WalletPayment (chacune @dataclass avec leurs champs propres).
# @dataclass OrderItem avec sku, quantity, unit_price: Money.
# @dataclass Order avec id, items: list[OrderItem], total: Money calculé via __post_init__, et method: PaymentMethod.

# Validation — ce code doit fonctionner :

# items = [OrderItem(sku="A", quantity=2, unit_price=Money(500, "EUR"))]
# order = Order(id="O-1", items=items, method=CreditCardPayment(card="4242..."))
# print(order.total)                  # Money(amount=1000, currency='EUR')

# tx_id = order.method.charge(order.total)
# assert order.method.refund(tx_id) is True

from dataclasses import dataclass
from abc import ABC, abstractmethod
from uuid import UUID

@dataclass(frozen=True)
class Money:
    amount: int = 0 # ? in cents
    currency: str = "EUR"

    def __add__(self, other: "Money"):
        self = replace(self, amount = self.amount + other.amount)

@dataclass
class PaymentMethod(ABC):
    @abstractmethod
    def charge(self, amount: Money) -> str:
        ...

    @abstractmethod
    def refund(self, transaction_id: str) -> bool:
        ...

@dataclass
class CreditCardPayment(PaymentMethod):
    card: UUID

    def charge(self, amount: Money) -> str:
        return ""
    def refund(self, transaction_id: UUID) -> bool:
        return True

@dataclass
class BankTransferPayment(PaymentMethod):
    pass

@dataclass
class WalletPayment(PaymentMethod):
    pass


@dataclass
class OrderItem:
    sku: str
    quantity: int
    unit_price: Money

@dataclass
class Order:
    id: str
    items: list[OrderItem]
    method: PaymentMethod
    total: Money = field(default=Money())

    def __post_init__(self):
        for i in items:
            self.total += i.unit_price

items = [OrderItem(sku="A", quantity=2, unit_price=Money(500, "EUR"))]
order = Order(id="O-1", items=items, method=CreditCardPayment(card="4242..."))
print(order.total)                  # Money(amount=1000, currency='EUR')

tx_id = order.method.charge(order.total)
assert order.method.refund(tx_id) is True

None


Le module M4 est validé lorsque :

- [x] L'apprenant peut convertir une classe "à la main" en `@dataclass` équivalente.
- [x] Les paramètres `default_factory`, `compare`, `repr` de `field()` sont maîtrisés.
- [x] L'apprenant connaît au moins trois raisons d'utiliser `frozen=True`.
- [x] Le piège du default mutable est identifié.
- [x] L'apprenant peut définir et utiliser une `abc.ABC` avec `@abstractmethod`.
- [x] Le mini-défi paiement passe les assertions de validation.
- [x] L'apprenant peut motiver le choix entre `dataclass`, `dataclass(frozen=True)`, `NamedTuple` et classe ordinaire.